test

In [2]:
import numpy as np
import pandas as pd

from ..hawkes_intertrade import (
    LogACDVolumeMLE,
    HawkesExpFixedDecayMLE,
    add_volume_buckets,
    build_side_volume_price_streams,
    prepare_trade_dataframe,
)


def test_hawkes_fixed_smoke():
    mu = np.array([0.2, 0.15])
    alpha = np.array([[0.1, 0.05], [0.04, 0.1]])
    beta = np.ones((2, 2))
    events = HawkesExpFixedDecayMLE.simulate(mu, alpha, beta, 30.0, seed=1)
    model = HawkesExpFixedDecayMLE(beta, max_iter=200).fit(events, end_times=30.0)
    assert model.baseline_.shape == (2,)
    assert model.adjacency_.shape == (2, 2)
    assert np.isfinite(model.log_likelihood_)


ImportError: attempted relative import with no known parent package

In [ ]:
def test_data_and_acd_smoke():
    df = pd.DataFrame({
        "timestamp": np.arange(20, dtype=float),
        "price": 100 + np.cumsum(np.random.default_rng(1).normal(0, 0.01, 20)),
        "volume": np.arange(1, 21, dtype=float),
    })
    out = prepare_trade_dataframe(df)
    out = add_volume_buckets(out, n_buckets=3)
    events, names = build_side_volume_price_streams(out)
    assert len(events) == len(names)
    valid = out["duration"].notna()
    acd = LogACDVolumeMLE(max_iter=100).fit(out.loc[valid, "duration"], out.loc[valid, "log_volume_z"])
    assert np.isfinite(acd.log_likelihood_)
